# Parent Document Retriever — 작은 청크로 검색, 큰 청크로 반환

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **ParentDocumentRetriever**가 왜 필요한지, 검색 정확도와 컨텍스트 풍부함의 트레이드오프 이해하기
- `parent_splitter`와 `child_splitter`의 역할 구분하기
- `InMemoryStore`로 부모-자식 문서 관계를 저장하는 방법 익히기

## 사전 지식

- 청크(Chunk) 크기가 검색 정확도와 컨텍스트에 어떻게 영향을 미치는지 이해
- VectorStoreRetriever 기본 사용법

---

```mermaid
flowchart TD
    D[원본 문서]:::input --> PS[parent_splitter<br/>큰 청크 생성<br/>1500자]:::process
    PS --> PD[(부모 문서<br/>InMemoryStore)]:::storage
    D --> CS[child_splitter<br/>작은 청크 생성<br/>300자]:::process
    CS --> VD[(자식 벡터<br/>VectorStore)]:::storage
    Q[사용자 쿼리]:::input --> VD
    VD -- 자식으로 검색 --> R[부모 문서 반환<br/>풍부한 컨텍스트]:::output
    PD --> R

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

## 핵심 아이디어

| 상황 | 문제 |
|------|------|
| 작은 청크만 사용 | 검색은 정확하지만 컨텍스트 부족 |
| 큰 청크만 사용 | 컨텍스트는 풍부하지만 검색 정확도 낮음 |
| **ParentDocument** | **작은 청크로 검색 + 부모 문서 반환** |

> **실무 팁**: 자식 청크 크기는 300~500자, 부모 청크 크기는 1000~2000자가 일반적으로 좋은 출발점이에요.

> 🔑 **핵심 개념**: 임베딩의 정확도와 LLM에 제공하는 컨텍스트 풍부함은 트레이드오프 관계입니다. ParentDocumentRetriever는 두 목표를 분리해서 각각 최적화합니다. 작은 청크로 정밀하게 검색하고, 부모 문서로 충분한 컨텍스트를 제공해요.

> 🎯 **강의 포인트**: InMemoryStore는 개발/학습용이고, 실제 프로덕션에서는 `LocalFileStore`나 `RedisStore` 등 영속적인 저장소를 사용해야 합니다. 서버를 재시작하면 InMemoryStore 데이터는 사라져요.

In [1]:
from dotenv import load_dotenv
load_dotenv()


True

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from langchain.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

# 문서 로드
loader = TextLoader("./data/ai-story.txt")
documents = loader.load()

# 두 가지 splitter 설정
# parent_splitter: 반환할 큰 청크 (컨텍스트 풍부)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=100)
# child_splitter: 검색에 사용할 작은 청크 (정확도 높음)
# 🎯 강의 포인트: child 크기는 parent의 1/5 ~ 1/3 비율이 일반적
child_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)

# Vectorstore 및 Docstore
# vectorstore: 자식 청크 임베딩 저장 (검색용)
# InMemoryStore: 부모 문서 원문 저장 (반환용)
vectorstore = FAISS.from_texts(["init"], OpenAIEmbeddings(model="text-embedding-3-small"))
store = InMemoryStore()

# ParentDocumentRetriever 생성
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# 문서 추가
parent_retriever.add_documents(documents)

print("✅ ParentDocumentRetriever 생성 완료")
print(f"  - 부모 청크 크기: 1500자")
print(f"  - 자식 청크 크기: 300자 (검색용)")

✅ ParentDocumentRetriever 생성 완료
  - 부모 청크 크기: 1500자
  - 자식 청크 크기: 300자 (검색용)


In [3]:
# ---------------------------------------------------
# ParentDocumentRetriever 검색 실행 및 반환 문서 크기 확인
# ---------------------------------------------------

# ============================================================
# TODO: parent_retriever.invoke(query)로 검색하고 반환된 문서 길이를 확인하세요
# 힌트: 반환 문서는 자식(300자)이 아닌 부모(1500자) 크기여야 함
# 예상 결과: 각 문서 길이가 1500자 내외로 출력
# ============================================================

# 검색
query = "Word2Vec의 작동 원리는?"
docs = parent_retriever.invoke(query)

print(f"📝 쿼리: {query}\n")
print("="*80)
print("[ParentDocumentRetriever 결과]")
print("="*80)
for i, doc in enumerate(docs, 1):
    print(f"[문서 {i}]")
    print(f"길이: {len(doc.page_content)}자 ← 부모 문서 (큰 청크)")
    print(f"내용: {doc.page_content[:200]}...")
    print("-"*80)

print("\n💡 장점:")
print("  - 작은 청크로 검색 → 정확도 높음")
print("  - 부모 문서 반환 → 컨텍스트 풍부")
print("  - 최상의 균형 달성")

📝 쿼리: Word2Vec의 작동 원리는?

[ParentDocumentRetriever 결과]
[문서 1]
길이: 1279자 ← 부모 문서 (큰 청크)
내용: ARIMA (자기회귀 누적 이동평균)

ARIMA 모델은 비계절적 시계열 데이터를 분석하고 예측하기 위해 설계되었습니다. 이 모델은 세 가지 주요 구성 요소를 기반으로 합니다: 자기회귀(AR) 부분, 차분(I) 부분, 그리고 이동평균(MA) 부분. AR 부분은 이전 관측값의 영향을 모델링하며, I 부분은 데이터를 안정화하는 데 필요한 차분의 횟수를 나타냅니...
--------------------------------------------------------------------------------
[문서 2]
길이: 698자 ← 부모 문서 (큰 청크)
내용: Word2Vec은 크게 두 가지 모델 아키텍처로 구성됩니다: Continuous Bag-of-Words (CBOW)와 Skip-Gram입니다. CBOW 모델은 주변 단어(맥락)를 기반으로 특정 단어를 예측하는 반면, Skip-Gram 모델은 하나의 단어로부터 주변 단어들을 예측합니다. 두 모델 모두 딥러닝이 아닌, 단순화된 신경망 구조를 사용하여 대규모 텍...
--------------------------------------------------------------------------------

💡 장점:
  - 작은 청크로 검색 → 정확도 높음
  - 부모 문서 반환 → 컨텍스트 풍부
  - 최상의 균형 달성


In [4]:
# ---------------------------------------------------
# 검색 방식 3방향 비교: 작은 청크 vs 큰 청크 vs ParentDocument
# ---------------------------------------------------

# 동일 문서를 3가지 방식으로 검색하여 결과 비교
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

compare_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 방식 1: 작은 청크만 (300자) — 검색 정확, 컨텍스트 부족
small_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
small_docs = small_splitter.split_documents(documents)
small_vs = FAISS.from_documents(small_docs, compare_embeddings)
small_retriever = small_vs.as_retriever(search_kwargs={"k": 2})

# 방식 2: 큰 청크만 (1500자) — 컨텍스트 풍부, 검색 정확도 낮음
big_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=100)
big_docs = big_splitter.split_documents(documents)
big_vs = FAISS.from_documents(big_docs, compare_embeddings)
big_retriever = big_vs.as_retriever(search_kwargs={"k": 2})

# 방식 3: ParentDocumentRetriever (위에서 생성한 parent_retriever 재사용)

compare_query = "Word2Vec의 작동 원리는?"

small_results = small_retriever.invoke(compare_query)
big_results = big_retriever.invoke(compare_query)
parent_results = parent_retriever.invoke(compare_query)

print(f"📝 쿼리: {compare_query}\n")
print(f"{'='*70}")
print(f"📊 검색 방식 3방향 비교")
print(f"{'='*70}")
print(f"{'방식':<30} {'문서수':>6} {'평균 길이':>10} {'총 문자수':>10}")
print(f"{'-'*60}")

for label, results in [
    ("작은 청크 (300자)", small_results),
    ("큰 청크 (1500자)", big_results),
    ("ParentDocument", parent_results),
]:
    n = len(results)
    total = sum(len(d.page_content) for d in results)
    avg = total // n if n > 0 else 0
    print(f"{label:<30} {n:>6} {avg:>10} {total:>10}")

print(f"\n{'='*70}")
print("[작은 청크 결과] — 정확하지만 컨텍스트 부족")
print(f"{'='*70}")
for i, doc in enumerate(small_results, 1):
    print(f"  [{i}] {len(doc.page_content):>4}자 | {doc.page_content[:60]}...")

print(f"\n{'='*70}")
print("[ParentDocument 결과] — 작은 청크로 검색 + 부모 문서 반환")
print(f"{'='*70}")
for i, doc in enumerate(parent_results, 1):
    print(f"  [{i}] {len(doc.page_content):>4}자 | {doc.page_content[:60]}...")

# child가 실제로 매칭한 부분 확인
print(f"\n{'='*70}")
print("🔍 child 청크가 실제로 매칭한 부분 확인")
print(f"{'='*70}")
# vectorstore에서 직접 similarity_search로 child 청크 확인
child_matches = vectorstore.similarity_search(compare_query, k=3)
for i, doc in enumerate(child_matches, 1):
    print(f"  [child {i}] {len(doc.page_content):>4}자 | {doc.page_content[:80]}...")

print("\n💡 관찰 포인트:")
print("  - child 청크(작은 크기)로 정밀 검색 → 반환은 부모(큰 크기)")
print("  - 작은 청크 단독 검색과 비슷한 정확도 + 큰 청크의 풍부한 컨텍스트")

📝 쿼리: Word2Vec의 작동 원리는?

📊 검색 방식 3방향 비교
방식                                문서수      평균 길이      총 문자수
------------------------------------------------------------
작은 청크 (300자)                        2        241        483
큰 청크 (1500자)                        2        988       1977
ParentDocument                      2        988       1977

[작은 청크 결과] — 정확하지만 컨텍스트 부족
  [1]  253자 | Word2Vec

Word2Vec은 자연어 처리(NLP) 분야에서 널리 사용되는 획기적인 단어 임베딩 기법 ...
  [2]  230자 | Word2Vec은 크게 두 가지 모델 아키텍처로 구성됩니다: Continuous Bag-of-Words (C...

[ParentDocument 결과] — 작은 청크로 검색 + 부모 문서 반환
  [1] 1279자 | ARIMA (자기회귀 누적 이동평균)

ARIMA 모델은 비계절적 시계열 데이터를 분석하고 예측하기 위해 설...
  [2]  698자 | Word2Vec은 크게 두 가지 모델 아키텍처로 구성됩니다: Continuous Bag-of-Words (C...

🔍 child 청크가 실제로 매칭한 부분 확인
  [child 1]  253자 | Word2Vec

Word2Vec은 자연어 처리(NLP) 분야에서 널리 사용되는 획기적인 단어 임베딩 기법 중 하나입니다. 2013년 Googl...
  [child 2]  230자 | Word2Vec은 크게 두 가지 모델 아키텍처로 구성됩니다: Continuous Bag-of-Words (CBOW)와 Skip-Gram입니다. ...
  [child 3]  234자 | Word2Vec의 벡터 표현은 다양한

---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 아이디어 | 작은 청크로 검색(정밀도), 부모 문서로 반환(컨텍스트) |
| 필수 구성 요소 | `child_splitter`, `parent_splitter` (선택), `InMemoryStore`, VectorStore |
| 두 가지 모드 | 전체 문서 반환 / 큰 청크 반환 (`parent_splitter` 유무로 결정) |
| 저장소 역할 | VectorStore — 자식 청크 임베딩 / InMemoryStore — 부모 문서 원문 |
| 성능 트레이드오프 | 인덱싱 시간·메모리 증가, 검색 품질 향상 |

### 파라미터 선택 가이드

| 상황 | 권장 설정 |
|------|-----------|
| 문서가 길고 정밀 검색이 필요할 때 | `child_chunk_size=200`, `parent_chunk_size=2000` |
| 문서가 짧고 전체 문맥이 중요할 때 | `parent_splitter=None` (전체 문서 반환 모드) |
| 메모리가 제한될 때 | `InMemoryStore` 대신 `LocalFileStore` 사용 |

### 다음 노트북 예고

**06-MultiQueryRetriever** — 단일 쿼리 하나로는 놓칠 수 있는 문서를, LLM이 여러 개의 쿼리를 자동 생성해 다각도로 검색하는 방법을 배워요.